# 🌳 Project 14.2 — Robot Fleet Tree & Network Navigator
**DSA for Mechatronics · Week 14 — Review & Final Project**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.

**Topics integrated:** Binary Trees (W07) · BFS/DFS Graphs (W09) · Heaps (W08) · Hash Maps (W05)


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq, math
from collections import defaultdict, deque, Counter, OrderedDict
from functools import lru_cache
import bisect

_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A robot fleet is organised as a binary tree of command nodes and connected via a
sensor network graph. We need four operations:

1. **Serialize and deserialize** the command tree — encode it as a string for
   transmission, then reconstruct it exactly (preorder with null markers)
2. **Level-order traversal** — inspect all nodes level by level (BFS on tree)
   and compute the maximum value at each level
3. **k-th largest reading** — find the k-th largest sensor reading without
   full sorting (min-heap of size k)
4. **BFS shortest path** — find the shortest hop-count path between two
   nodes in the unweighted sensor network graph


## Step 1 — Build command tree and sensor network

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_TREE   = random.randint(10, 16)
N_NET    = random.randint(8, 12)
NET_P    = random.uniform(0.30, 0.50)
N_RDGS   = random.randint(15, 22)
K_LARGE  = random.randint(2, 5)

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val   = val
        self.left  = left
        self.right = right

def build_random_tree(n, val_range=(1, 50)):
    """Build a random binary tree with n nodes using level-order insertion."""
    if n == 0: return None
    vals  = random.sample(range(val_range[0], val_range[1]+1), n)
    root  = TreeNode(vals[0])
    queue = deque([root])
    i     = 1
    while i < n:
        node = queue.popleft()
        if i < n:
            node.left  = TreeNode(vals[i]); i += 1
            queue.append(node.left)
        if i < n:
            node.right = TreeNode(vals[i]); i += 1
            queue.append(node.right)
    return root

TREE_ROOT = build_random_tree(N_TREE)
RDGS      = [random.randint(1, 100) for _ in range(N_RDGS)]

# Sensor network graph (undirected, unweighted)
NET_NODES = list(range(N_NET))
NET_EDGES = []
# Guarantee connected via spanning chain
shuffled  = list(range(N_NET))
random.shuffle(shuffled)
for i in range(1, N_NET):
    NET_EDGES.append((shuffled[i-1], shuffled[i]))
for u in range(N_NET):
    for v in range(u+1, N_NET):
        if random.random() < NET_P and (u,v) not in NET_EDGES and (v,u) not in NET_EDGES:
            NET_EDGES.append((u, v))
NET_ADJ = defaultdict(list)
for u, v in NET_EDGES:
    NET_ADJ[u].append(v)
    NET_ADJ[v].append(u)

NET_SRC = 0
NET_DST = N_NET - 1

def tree_to_list(root):
    """Convert tree to level-order list (None for missing nodes) for display."""
    if not root: return []
    result = []; queue = deque([root])
    while queue:
        node = queue.popleft()
        if node:
            result.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append(None)
    while result and result[-1] is None:
        result.pop()
    return result

print(f"Command tree (n={N_TREE} nodes):")
print(f"  Level-order : {tree_to_list(TREE_ROOT)}")
print()
print(f"Sensor readings (n={N_RDGS}): {RDGS}")
print(f"  Find k={K_LARGE}-th largest")
print()
print(f"Sensor network: {N_NET} nodes, {len(NET_EDGES)} edges")
print(f"  Edges: {NET_EDGES[:10]}{'...' if len(NET_EDGES)>10 else ''}")
print(f"  Route: {NET_SRC} → {NET_DST}")


## Step 2 — Serialize and deserialize binary tree

In [ ]:
def serialize(root):
    """
    Serialize tree to string using PREORDER traversal with null markers.

    Visit root → left subtree → right subtree.
    Use "#" for None nodes — these are essential to reconstruct the shape.
    Separate tokens with commas.

    Why preorder? The root always comes first, making deserialization
    straightforward: read root, then recursively build left and right subtrees.

    O(n) time and space.
    """
    tokens = []
    def dfs(node):
        if node is None:
            tokens.append("#")
            return
        tokens.append(str(node.val))
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return ",".join(tokens)

def deserialize(data):
    """
    Deserialize from the preorder string produced by serialize().

    Use an iterator over the token list.
    At each call: read next token.
      - If "#" → return None (leaf boundary).
      - Otherwise → create node with that value, then recursively
        build left subtree (next tokens) and right subtree (following tokens).

    O(n) time.
    """
    it = iter(data.split(","))
    def build():
        tok = next(it)
        if tok == "#": return None
        node = TreeNode(int(tok))
        node.left  = build()
        node.right = build()
        return node
    return build()

def trees_equal(a, b):
    if a is None and b is None: return True
    if a is None or b is None:  return False
    return a.val == b.val and trees_equal(a.left, b.left) and trees_equal(a.right, b.right)

serialized    = serialize(TREE_ROOT)
reconstructed = deserialize(serialized)
rt_ok         = trees_equal(TREE_ROOT, reconstructed)
token_count   = len(serialized.split(","))
null_count    = serialized.count("#")

print(f"Serialize / Deserialize (preorder + null markers):")
print()
print(f"  Original (level-order) : {tree_to_list(TREE_ROOT)}")
print(f"  Serialized string      : {serialized[:80]}{'...' if len(serialized)>80 else ''}")
print(f"  Token count            : {token_count}  (n={N_TREE} nodes + {null_count} null markers)")
print(f"  Reconstructed          : {tree_to_list(reconstructed)}")
print(f"  Round-trip correct     : {'✅ YES' if rt_ok else '❌ NO'}")
print()
print(f"  Note: 2n+1 tokens total for n nodes — each node generates")
print(f"  exactly 2 null children at the leaves: {2*N_TREE+1} expected, got {token_count}")


## Step 3 — Level-order traversal: BFS on tree

In [ ]:
def level_order(root):
    """
    Level-order traversal (BFS on tree).

    Use a queue (deque). Process one level at a time:
      1. Snapshot the current queue length = nodes on this level.
      2. Dequeue and record all nodes of this level.
      3. Enqueue their children (next level).

    Returns: list of levels, where each level is a list of node values.
    O(n) time and space.
    """
    if not root: return []
    levels = []
    queue  = deque([root])
    while queue:
        level_size = len(queue)
        level_vals = []
        for _ in range(level_size):
            node = queue.popleft()
            level_vals.append(node.val)
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)
        levels.append(level_vals)
    return levels

levels      = level_order(TREE_ROOT)
level_maxes = [max(lv) for lv in levels]
level_mins  = [min(lv) for lv in levels]
tree_height = len(levels)

print(f"Level-order traversal (tree height = {tree_height}):")
print()
for i, lv in enumerate(levels):
    print(f"  Level {i}: {lv}  → max={max(lv):3d}  min={min(lv):3d}")
print()
print(f"  Tree height      : {tree_height}")
print(f"  Level maxima     : {level_maxes}")
print(f"  Level minima     : {level_mins}")
print(f"  Global max       : {max(level_maxes)}")
print(f"  Global min       : {min(level_mins)}")
all_nodes_bfs = [v for lv in levels for v in lv]
all_nodes_expected = tree_to_list(TREE_ROOT)
print(f"  BFS order correct: {'✅ YES' if all_nodes_bfs == all_nodes_expected else '❌ NO'}")


## Step 4 — k-th largest reading (min-heap of size k)

In [ ]:
def kth_largest(nums, k):
    """
    Find the k-th largest element without full sorting.

    Maintain a MIN-HEAP of size k:
      - The heap always holds the k largest elements seen so far.
      - The heap's minimum (root) = the k-th largest overall.
      - For each new element:
          If it's larger than the heap root → push it, then pop the root
          (so heap stays size k and always holds the k largest).

    O(n log k) time — much faster than O(n log n) full sort when k << n.
    O(k) space.
    """
    heap = nums[:k]
    heapq.heapify(heap)
    for num in nums[k:]:
        if num > heap[0]:
            heapq.heapreplace(heap, num)
    return heap[0], sorted(heap, reverse=True)

kth_val, top_k = kth_largest(RDGS, K_LARGE)
sorted_desc    = sorted(RDGS, reverse=True)
verify_kth     = sorted_desc[K_LARGE - 1]

print(f"k-th largest reading (k={K_LARGE}, n={N_RDGS}):")
print(f"  Readings       : {RDGS}")
print()
print(f"  k={K_LARGE}-th largest  : {kth_val}")
print(f"  Verification   : sorted[{K_LARGE-1}] = {verify_kth}  "
      f"{'✅' if kth_val == verify_kth else '❌'}")
print(f"  Top-{K_LARGE} values  : {top_k}")
print(f"  Full sorted top-{K_LARGE}: {sorted_desc[:K_LARGE]}")
print()
print(f"  Complexity advantage: O(n log k) heap vs O(n log n) sort")
import math
ratio = N_RDGS * math.log2(N_RDGS) / (N_RDGS * math.log2(K_LARGE)) if K_LARGE > 1 else 1
print(f"  log(n)/log(k) speedup factor ≈ {ratio:.2f}×")


## Step 5 — BFS shortest path in sensor network

In [ ]:
def bfs_shortest_path(adj, src, dst, n_nodes):
    """
    BFS shortest path in an UNWEIGHTED undirected graph.

    BFS explores nodes in order of increasing hop distance.
    When dst is first dequeued, the distance recorded is the shortest.

    Algorithm:
      1. dist[src] = 0; push src to queue.
      2. For each dequeued node u:
         For each neighbour v of u:
           If v not visited → dist[v] = dist[u] + 1; enqueue v.
      3. Reconstruct path by backtracking via prev[].

    O(V + E) time and space.
    NOTE: For weighted graphs, use Dijkstra (W13).
          BFS gives incorrect distances if edge weights differ.
    """
    dist = [-1] * n_nodes
    prev = [-1] * n_nodes
    dist[src] = 0
    queue = deque([src])
    while queue:
        u = queue.popleft()
        if u == dst: break
        for v in adj[u]:
            if dist[v] == -1:
                dist[v]  = dist[u] + 1
                prev[v]  = u
                queue.append(v)
    if dist[dst] == -1:
        return -1, []
    path = []
    node = dst
    while node != -1:
        path.append(node)
        node = prev[node]
    path.reverse()
    return dist[dst], path

net_dist, net_path = bfs_shortest_path(NET_ADJ, NET_SRC, NET_DST, N_NET)

print(f"BFS shortest path (unweighted, {NET_SRC} → {NET_DST}):")
print()
print(f"  Network edges : {NET_EDGES}")
print()
if net_dist >= 0:
    print(f"  Shortest hops : {net_dist}")
    print(f"  Path          : {' → '.join(map(str, net_path))}")
    # Verify path is valid
    valid_path = all((net_path[i], net_path[i+1]) in NET_EDGES or
                     (net_path[i+1], net_path[i]) in NET_EDGES
                     for i in range(len(net_path)-1))
    print(f"  Path valid    : {'✅ YES' if valid_path else '❌ NO'}")
    print(f"  Hops = edges  : {len(net_path)-1} == {net_dist}  "
          f"{'✅' if len(net_path)-1 == net_dist else '❌'}")
else:
    print(f"  No path from {NET_SRC} to {NET_DST}")
    valid_path = True; net_dist = -1


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " ROBOT FLEET TREE & NETWORK NAVIGATOR — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  SERIALIZE / DESERIALIZE TREE" + " "*(W-30) + "║")
print(f"║  {'Tree nodes':<28}: {N_TREE:<26}║")
print(f"║  {'Token count':<28}: {token_count} ({null_count} null markers){'':<6}║")
print(f"║  {'Round-trip correct':<28}: {'YES ✅' if rt_ok else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  LEVEL-ORDER TRAVERSAL (BFS)" + " "*(W-29) + "║")
print(f"║  {'Tree height':<28}: {tree_height:<26}║")
print(f"║  {'Level maxima':<28}: {str(level_maxes):<26}║")
print(f"║  {'Global max / min':<28}: {max(level_maxes)} / {min(level_mins)}{'':<18}║")
print("╠" + "═"*W + "╣")
print("║  K-TH LARGEST (MIN-HEAP)" + " "*(W-25) + "║")
print(f"║  {'Readings (n)':<28}: {N_RDGS:<26}║")
print(f"║  {'k':<28}: {K_LARGE:<26}║")
print(f"║  {'k-th largest':<28}: {kth_val:<26}║")
print(f"║  {'Correct':<28}: {'YES ✅' if kth_val == verify_kth else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  BFS SHORTEST PATH" + " "*(W-19) + "║")
print(f"║  {'Network nodes':<28}: {N_NET:<26}║")
print(f"║  {'Network edges':<28}: {len(NET_EDGES):<26}║")
if net_dist >= 0:
    print(f"║  {'Shortest hops':<28}: {net_dist:<26}║")
    print(f"║  {'Path':<28}: {' → '.join(map(str,net_path)):<26}║")
    print(f"║  {'Path valid':<28}: {'YES ✅' if valid_path else 'NO ❌':<26}║")
else:
    print(f"║  {'Shortest hops':<28}: {'No path':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What do they tell you about the algorithms in this project?

*Your answer here:*

---

**Q2 — Which technique was most surprising, and why?**
Pick one algorithm from this project. Explain why the approach works — and give one scenario where it would *fail* or need modification.

*Your answer here:*

---

**Q3 — How does this project connect to earlier weeks?**
Name at least two specific weeks whose topics appear in this project and explain the connection.

*Your answer here:*
